In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\| **Quickstart** \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Quickstart
==========

This section runs through the API for common tasks in machine learning.
Refer to the links in each section to dive deeper.

Working with data
-----------------

PyTorch has two [primitives to work with
data](https://pytorch.org/docs/stable/data.html):
`torch.utils.data.DataLoader` and `torch.utils.data.Dataset`. `Dataset`
stores the samples and their corresponding labels, and `DataLoader`
wraps an iterable around the `Dataset`.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

PyTorch offers domain-specific libraries such as
[TorchText](https://pytorch.org/text/stable/index.html),
[TorchVision](https://pytorch.org/vision/stable/index.html), and
[TorchAudio](https://pytorch.org/audio/stable/index.html), all of which
include datasets. For this tutorial, we will be using a TorchVision
dataset.

The `torchvision.datasets` module contains `Dataset` objects for many
real-world vision data like CIFAR, COCO ([full list
here](https://pytorch.org/vision/stable/datasets.html)). In this
tutorial, we use the FashionMNIST dataset. Every TorchVision `Dataset`
includes two arguments: `transform` and `target_transform` to modify the
samples and labels respectively.


In [ ]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

In [ ]:
training_data

We pass the `Dataset` as an argument to `DataLoader`. This wraps an
iterable over our dataset, and supports automatic batching, sampling,
shuffling and multiprocess data loading. Here we define a batch size of
64, i.e. each element in the dataloader iterable will return a batch of
64 features and labels.


In [ ]:
from torch.utils.data import Dataset
from sg.models import Encoder


class EncoderDataset(Dataset):
    def __init__(self, subj_id, sess_id, **kwargs):
        self.subj_id = subj_id
        self.sess_id = sess_id

        self.encoder = Encoder(subj_id=subj_id, sess_id=sess_id, **kwargs)
        self.encoder.build_dm()

        self.tvs = self.encoder.tvs
        self.robs = self.encoder.robs.astype("float32")

    def __len__(self):
        return self.encoder.num_trials

    def __getitem__(self, idx):
        return self.tvs[idx], self.robs[idx]

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"
data = EncoderDataset(subj_id, sess_id)

In [ ]:
batch_size = 64
dataloader = DataLoader(data, batch_size=batch_size)

for X, y in dataloader:
    print(f"[N, D]: {X.shape}")

In [ ]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Read more about [loading data in PyTorch](data_tutorial.html).


------------------------------------------------------------------------


Creating Models
===============

To define a neural network in PyTorch, we create a class that inherits
from
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
We define the layers of the network in the `__init__` function and
specify how data will pass through the network in the `forward`
function. To accelerate operations in the neural network, we move it to
the
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [ ]:
# encoder and drift weights is ultimately just a linear layer, tents and tv concat together (but fit apart)

In [ ]:
encoder = Encoder(subj_id, sess_id)
encoder.verify()

In [ ]:
# TODO: regularization and weight tying
class AE(nn.Module):
    def __init__(self, subj_id, sess_id, k=10, **kwargs):
        super().__init__()

        self.encoder = Encoder(subj_id, sess_id, **kwargs)
        self.encoder.build_dm()

        self.k = k

        self.e = nn.Sequential(
            nn.Linear(
                in_features=self.encoder.num_units, out_features=self.k, device=device
            ),
            nn.ReLU(),
        )

        self.d = nn.Linear(
            in_features=self.k, out_features=self.encoder.num_units, device=device
        )

        self.criterion = nn.MSELoss()

    def forward(self, x):
        z = self.e(x)
        x_hat = self.d(z)
        return x_hat

    def loss(self, x):
        x_hat = self(x)
        return self.criterion(x, x_hat)

In [ ]:
from torch import optim
import numpy as np


def fit_ae(model, dataloader, num_epochs=20, device="cpu"):
    losses = []
    optimizer = optim.LBFGS(model.parameters())

    for epoch in range(num_epochs):
        total_loss = 0
        for _, robs in dataloader:
            # Move images to device
            robs = robs.to(device)

            def closure():
                optimizer.zero_grad()
                loss = model.loss(robs)
                loss.backward()
                return loss

            loss = optimizer.step(closure)
            total_loss += loss.item()

        # Print epoch statistics
        avg_loss = total_loss / len(dataloader)
        losses.append(avg_loss)
        print(f"Epoch [{epoch + 1}/{num_epochs}], Average Loss: {avg_loss:.4f}")

        # early stopping
        if epoch > 5 and -1 * np.diff(losses)[-3:-1].mean() < 1e-1:
            break


device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"using {device}")

ae = AE(subj_id, sess_id, k=10)
fit_ae(ae, dataloader, num_epochs=10, device=device)

In [ ]:
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")


# Define model
class OffsetLinearReLU(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        c,
    ):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, device=device)
        self.relu = nn.ReLU()
        self.c = c

    def forward(self, x):
        return self.relu(self.c + self.linear(x))


class LatentNet(nn.Module):
    def __init__(
        self,
        in_features,
        k,
    ):
        super().__init__()
        self.e = OffsetLinearReLU(in_features, k, c=1)
        self.d = nn.Linear(k, in_features, device=device)

    def forward(self, x):
        return self.d(self.e(x))


class SharedGain(nn.Module):
    def __init__(self, subj_id, sess_id, num_latents=3, **kwargs):
        super().__init__()

        self.subj_id = subj_id
        self.sess_id = sess_id
        self.num_latents = num_latents
        self.kwargs = kwargs

        self.__init_encoder__()
        self.__init_ae__()

        self.tv_net = nn.Linear(
            in_features=self.encoder.num_tv,
            out_features=self.encoder.num_units,
        )
        self.drift_net = nn.Linear(
            in_features=self.encoder.num_tents,
            out_features=self.encoder.num_units,
        )
        self.latent_net = LatentNet(
            in_features=self.encoder.num_units,
            k=self.num_latents,
        )

        self.__init_weights__()

    def __init_encoder__(self):
        self.encoder = Encoder(self.subj_id, self.sess_id, **self.kwargs)
        self.encoder.fit_encoder()

    def __init_ae__(self):
        self.ae = AE(self.subj_id, self.sess_id, k=self.num_latents)
        self.dataloader = DataLoader(
            EncoderDataset(self.subj_id, self.sess_id), batch_size=64
        )
        fit_ae(self.ae, self.dataloader, num_epochs=10, device=device)

    def __init_weights__(self):
        # init tv network weights to those found by the encoder
        self.tv_net.weight = nn.Parameter(torch.tensor(self.encoder.encoder_weights))
        self.tv_net.bias = nn.Parameter(torch.tensor(self.encoder.encoder.intercept_))

        # and the drift network too
        self.drift_net.weight = nn.Parameter(
            torch.tensor(self.encoder.baseline_weights)
        )
        self.drift_net.bias = nn.Parameter(
            torch.tensor(self.encoder.baseline_model.intercept_)
        )

        # and finally the latent network
        self.latent_net.e.linear.weight = nn.Parameter(
            torch.tensor(self.ae.e[0].weight)
        )
        self.latent_net.e.linear.bias = nn.Parameter(torch.tensor(self.ae.e[0].bias))

        self.latent_net.d.weight = nn.Parameter(torch.tensor(self.ae.d.weight))
        self.latent_net.d.bias = nn.Parameter(torch.tensor(self.ae.d.bias))

    def forward(self, x):
        xhat_tv = self.tv_net(x)
        xhat_drift = self.drift_net(x)
        xhat_latent = self.latent_net(x)

        xhat = xhat_latent * xhat_tv + xhat_drift
        return xhat


model = SharedGain(subj_id, sess_id, num_latents=10).to(device)
print(model)

In [ ]:
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")


# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)
print(model)

Read more about [building neural networks in
PyTorch](buildmodel_tutorial.html).


------------------------------------------------------------------------


Optimizing the Model Parameters
===============================

To train a model, we need a [loss
function](https://pytorch.org/docs/stable/nn.html#loss-functions) and an
[optimizer](https://pytorch.org/docs/stable/optim.html).


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In a single training loop, the model makes predictions on the training
dataset (fed to it in batches), and backpropagates the prediction error
to adjust the model\'s parameters.


In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

We also check the model\'s performance against the test dataset to
ensure it is learning.


In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(
        f"Test Error: \n Accuracy: {(100 * correct):>0.1f}%, Avg loss: {test_loss:>8f} \n"
    )

The training process is conducted over several iterations (*epochs*).
During each epoch, the model learns parameters to make better
predictions. We print the model\'s accuracy and loss at each epoch;
we\'d like to see the accuracy increase and the loss decrease with every
epoch.


In [ ]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Read more about [Training your model](optimization_tutorial.html).


------------------------------------------------------------------------


Saving Models
=============

A common way to save a model is to serialize the internal state
dictionary (containing the model parameters).


In [ ]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Loading Models
==============

The process for loading a model includes re-creating the model structure
and loading the state dictionary into it.


In [ ]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

This model can now be used to make predictions.


In [ ]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Read more about [Saving & Loading your
model](saveloadrun_tutorial.html).
